## Homework #2

## 1. 전처리

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", as_frame=False)

X, y = mnist.data, mnist.target

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
import matplotlib.pyplot as plt

def plot_digit(image_data):
    image = image_data.reshape(28, 28)
    plt.imshow(image, cmap="binary")
    plt.axis("off")

some_digit = X[1]
plot_digit(some_digit)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

## 2. Binary classifier

In [ ]:
from sklearn.linear_model import SGDClassifier

y_train_0 = (y_train == '0')
y_test_0 = (y_test == '0')

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_0)

sgd_clf.predict([some_digit])

## 3. Cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(sgd_clf, X_train, y_train_0, cv=3, scoring="accuracy")

## 4. Confussion matrix

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_0, cv=3)
cm = confusion_matrix(y_train_0, y_train_pred)

print(cm)

## 5. Performance measures

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_train_0, y_train_pred)
precision =  precision_score(y_train_0, y_train_pred)
recall = recall_score(y_train_0, y_train_pred)
f1 = f1_score(y_train_0, y_train_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1:.4f}")

## 6. ROC curve

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_scores = cross_val_predict(sgd_clf, X_train, y_train_0, cv=3,
                             method="decision_function")

precisions, recalls, thresholds_pr = precision_recall_curve(y_train_0, y_scores)
fpr, tpr, thresholds_roc = roc_curve(y_train_0, y_scores)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label="ROC curve")
plt.plot([0, 1], [0, 1], "k:", label="Random classifier's ROC curve")

plt.xlabel("False Positive Rate (Fall-Out)")
plt.ylabel("True Positive Rate (Recall)")
plt.grid()
plt.axis([0, 1, 0, 1])
plt.legend(loc="lower right", fontsize=13)
plt.show()

print("ROC AUC:", roc_auc_score(y_train_0, y_scores))

#### Random Forest classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score, precision_score, recall_score

forest_clf = RandomForestClassifier(random_state=42)

y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_0, cv=3, method="predict_proba")
y_scores_forest = y_probas_forest[:, 1]

precisions_forest, recalls_forest, thresholds_forest = (precision_recall_curve(y_train_0, y_scores_forest))

plt.figure(figsize=(6, 5))
plt.plot(recalls_forest, precisions_forest, "b-", linewidth=2, label="Random Forest")
plt.plot(recalls, precisions, "--", linewidth=2, label="SGD")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.axis([0, 1, 0, 1])
plt.grid()
plt.legend(loc="lower left")
plt.show()

y_train_pred_forest = cross_val_predict(forest_clf, X_train, y_train_0, cv=3)

print("F1:", f1_score(y_train_0, y_train_pred_forest))
print("Precision:", precision_score(y_train_0, y_train_pred_forest))
print("Recall:", recall_score(y_train_0, y_train_pred_forest))
print("ROC AUC:", roc_auc_score(y_train_0, y_scores_forest))

## 7. Multiclass classification

In [ ]:
from sklearn.svm import SVC

svm_clf = SVC(random_state=42)

svm_clf.fit(X_train[:2000], y_train[:2000])

print("예측:", svm_clf.predict([some_digit]))

some_digit_scores = svm_clf.decision_function([some_digit])

print("클래스별 점수:", some_digit_scores.round(2))
print("예측 클래스:", svm_clf.classes_[some_digit_scores.argmax()])

In [ ]:
from sklearn.multiclass import OneVsRestClassifier

ovr_clf = OneVsRestClassifier(SVC(random_state=42))
ovr_clf.fit(X_train[:2000], y_train[:2000])

print("예측:", ovr_clf.predict([some_digit]))
print("분류기 개수:", len(ovr_clf.estimators_))

In [ ]:
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train)
sgd_clf.predict([some_digit])

In [ ]:
cross_val_score(sgd_clf, X_train, y_train, cv=3, scoring="accuracy")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype("float64"))
cross_val_score(sgd_clf, X_train_scaled, y_train, cv=3, scoring="accuracy")

## 8. Error analysis

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_train_pred = cross_val_predict(sgd_clf, X_train_scaled, y_train, cv=3)
plt.rc('font', size=9)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred)
plt.show()

In [ ]:
plt.rc('font', size=10)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        normalize="true", values_format=".0%")
plt.show()

In [ ]:
sample_weight = (y_train_pred != y_train)
plt.rc('font', size=10)
ConfusionMatrixDisplay.from_predictions(y_train, y_train_pred,
                                        sample_weight=sample_weight,
                                        normalize="true", values_format=".0%")
plt.show()

In [ ]:
cl_a, cl_b = '3', '5'
X_aa = X_train[(y_train == cl_a) & (y_train_pred == cl_a)]
X_ab = X_train[(y_train == cl_a) & (y_train_pred == cl_b)]
X_ba = X_train[(y_train == cl_b) & (y_train_pred == cl_a)]
X_bb = X_train[(y_train == cl_b) & (y_train_pred == cl_b)]